# Nagelhus et al. (2023), single cells: changes between trials and object traces

**Paper.** Nagelhus et al. (2023), *Neuron* 111, 2091-2104,
[doi:10.1016/j.neuron.2023.04.008](https://doi.org/10.1016/j.neuron.2023.04.008); claims file written
from the version of record with its STAR Methods and Supplemental information.

**Dataset.** EBRAINS [10.25493/Q0FG-1X8](https://doi.org/10.25493/Q0FG-1X8), version v1, the version
whose DOI the paper cites and the only one published (see `01_single_cells.ipynb` for the data check
of every file).

| Claim | Tested or blocked |
|---|---|
| fig2-fraction-changed-no-o | tested |
| fig2d-o-om-change | tested |
| figs4-object-traces | tested |
| figs2d-traces-no-interference | tested |

The classification of the Methods (Spatial correlation and rate overlap): within-trial distributions
from the No Object trial, split 500 times into two random groups of 1 s sections; between-trial
distributions from a random half of each trial's 1 s sections, 500 times; a cell changes on a
measure when the mean between-trial value is below the 1st percentile of the within-trial values and
a signed-rank test (pairs by repetition, as the claims file says) gives p < 0.05. Maps are the
2.5 cm, speed-filtered, blanked single-cell maps, built as the claims file now gives (answered
2026-10-21: the rate per bin, unvisited bins 0, then smoothed and blanked; printed as `rm` b); rate
overlap uses speed-filtered rates (reading a) or all of the sections (b). The random groups are drawn once
per session and comparison and shared by its cells (a standard choice: they are random draws either
way).

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import objcode as oc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", 40)

started = time.time()
CA1, meta1 = oc.load(oc.SINGLE_FILES["CA1"])
CON1, metac = oc.load(oc.SINGLE_FILES["CON_CA1"])
print(f"dataset {oc.DATASET_ID} version {oc.DATASET_VERSION}: read in {time.time() - started:.0f} s")

In [ ]:
rows = []
for name, sess in (("CA1", CA1), ("CON_CA1", CON1)):
    for s in sess:
        rows.append({"file": name, "rat": s.rat, "units": s.n_units, "trials": "-".join(t.name for t in s.trials),
                     "sections NO or CON1 (1 s)": int(np.floor(s.trials[0].t[-1] + s.trials[0].dt - s.trials[0].t[0]))})
df = pd.DataFrame(rows)
print(df.groupby(["file", "trials"]).agg(sessions=("units", "size"), units=("units", "sum"),
                                         rats=("rat", "nunique")).to_string())
print("1 s sections in the first trial: min", df["sections NO or CON1 (1 s)"].min(), "median",
      int(df["sections NO or CON1 (1 s)"].median()), "max", df["sections NO or CON1 (1 s)"].max())
print("Cells with a No Object' trial:", int(df[(df.file == "CA1") & df.trials.str.contains("NOp")].units.sum()), "of 1189")

The CA1 file holds 1,189 cells in 60 sessions; 1,090 cells have a No Object' trial (44 sessions).
The paper says the fourth trial was missing for 8.5% of cells; here it is missing for 99 of 1,189
(8.3%).

In [ ]:
R = 500
PAIRS = [("NO", "O"), ("O", "OM"), ("NO", "NOp")]


def classify(sessions, rm, pairs):
    """Per cell: within and between distributions and the change decisions."""
    out = []
    for si, s in enumerate(sessions):
        NO = s.trial("NO")
        sm = {t.name: oc.SecMaps(t, range(s.n_units)) for t in s.trials}
        rng = np.random.default_rng(oc.seed_for("within", si))
        W1, W2 = oc.random_halves(sm["NO"].n, R, rng)
        prep_w = (sm["NO"].prepare(W1, rm), sm["NO"].prepare(W2, rm))
        prep_b = {}
        for a, b in pairs:
            if a not in sm or b not in sm:
                continue
            rng = np.random.default_rng(oc.seed_for("between", si, a, b))
            Ba, _ = oc.random_halves(sm[a].n, R, rng)
            Bb, _ = oc.random_halves(sm[b].n, R, rng)
            prep_b[a, b] = (sm[a].prepare(Ba, rm), sm[b].prepare(Bb, rm))
        for u in range(s.n_units):
            m1, f1, a1 = sm["NO"].groups(prep_w[0], u)
            m2, f2, a2 = sm["NO"].groups(prep_w[1], u)
            w_sc = oc.pearson_rows(m1, m2)
            w_ro = {"a": np.minimum(f1, f2) / np.maximum(f1, f2), "b": np.minimum(a1, a2) / np.maximum(a1, a2)}
            rec = {"session": si, "unit": u, "rat": s.rat}
            for (a, b), (pa, pb) in prep_b.items():
                x1, g1, h1 = sm[a].groups(pa, u)
                x2, g2, h2 = sm[b].groups(pb, u)
                b_sc = oc.pearson_rows(x1, x2)
                b_ro = {"a": np.minimum(g1, g2) / np.maximum(g1, g2), "b": np.minimum(h1, h2) / np.maximum(h1, h2)}
                sc, _ = oc.change_test(w_sc, b_sc)
                rec[f"{a}-{b} sc"] = sc
                for r in "ab":
                    ro, _ = oc.change_test(w_ro[r], b_ro[r])
                    rec[f"{a}-{b} ro_{r}"] = ro
                    rec[f"{a}-{b} any_{r}"] = sc or ro
                rec[f"{a}-{b} mean within sc"] = np.nanmean(w_sc)
                rec[f"{a}-{b} mean between sc"] = np.nanmean(b_sc)
            out.append(rec)
    return pd.DataFrame(out)


CLASS = {}
for rm in "b":
    t0 = time.time()
    CLASS[rm] = classify(CA1, rm, PAIRS)
    print(f"rm {rm}: classified {len(CLASS[rm])} cells in {time.time() - t0:.0f} s")

## fig2-fraction-changed-no-o

> About two thirds of CA1 cells (67.5%, 803 of 1,189) changed their spatial firing, their rate, or
> both between NO and O.

In [ ]:
rows = []
for rm in "b":
    C = CLASS[rm]
    for r in "ab":
        k = int(C[f"NO-O any_{r}"].sum())
        rows.append({"rm": rm, "rate overlap": r, "changed": k, "cells": len(C), "fraction": k / len(C),
                     "spatial only": int((C["NO-O sc"] & ~C[f"NO-O ro_{r}"]).sum()),
                     "rate only": int((~C["NO-O sc"] & C[f"NO-O ro_{r}"]).sum()),
                     "both": int((C["NO-O sc"] & C[f"NO-O ro_{r}"]).sum()),
                     "within 0.675+/-0.0675": abs(k / len(C) - 0.675) <= 0.0675})
fig2f = pd.DataFrame(rows)
print(fig2f.to_string(index=False))
C = CLASS["b"]
print("\nBy rat (rm b, rate overlap a): changed/cells")
print(C.groupby("rat")["NO-O any_a"].agg(["sum", "size"]).T.to_string())
print("\nMean within-trial and between-trial spatial correlation (rm b), median over cells:",
      round(C["NO-O mean within sc"].median(), 3), round(C["NO-O mean between sc"].median(), 3))

**Result: reproduced.** Between NO and O, 789 (66.4%, rate overlap a) and 793 (66.7%, rate overlap
b) of the 1,189 cells change, both within 0.675 ± 0.0675. With the rate maps the criterion now
gives, the two rate-overlap readings agree; before, one map reading put one of them outside. The
fraction changing differs between rats (34.6% in 25371 to 76.9% in 25450, printed above).

## fig2d-o-om-change

> Moving the object also changed spatial firing and rate more than in control trials, and 69.7% of
> cells (829 of 1,189) changed between O and OM.

In [ ]:
def pair_measures(sessions, a, b, rm):
    sc, roa, rob = [], [], []
    for s in sessions:
        ta, tb = s.trial(a), s.trial(b)
        ma, _ = oc.single_maps(ta, rm)
        mb, _ = oc.single_maps(tb, rm)
        for u in range(s.n_units):
            sc.append(oc.pearson_nan(ma[u], mb[u]))
            roa.append(oc.rate_overlap(oc.mean_rate(ta, u, True), oc.mean_rate(tb, u, True)))
            rob.append(oc.rate_overlap(oc.mean_rate(ta, u, False), oc.mean_rate(tb, u, False)))
    return pd.DataFrame({"sc": sc, "ro_a": roa, "ro_b": rob})


rows = []
for rm in "b":
    X = pair_measures(CA1, "O", "OM", rm)
    Y = pair_measures(CON1, "CON2", "CON3", rm)
    Cl = CLASS[rm]
    for r in "ab":
        z1, p1 = oc.ranksum(X.sc.dropna(), Y.sc.dropna())
        z2, p2 = oc.ranksum(X[f"ro_{r}"].dropna(), Y[f"ro_{r}"].dropna())
        k = int(Cl[f"O-OM any_{r}"].sum())
        parts = [p1 < 0.05 and X.sc.median() < Y.sc.median(), p2 < 0.05 and X[f"ro_{r}"].median() < Y[f"ro_{r}"].median(),
                 abs(k / len(Cl) - 0.697) <= 0.0697]
        rows.append({"rm": rm, "rate overlap": r, "sc median O-OM": X.sc.median(), "sc median CON2-CON3": Y.sc.median(),
                     "sc z": z1, "sc p": p1, "ro median O-OM": X[f"ro_{r}"].median(), "ro median CON": Y[f"ro_{r}"].median(),
                     "ro z": z2, "ro p": p2, "changed": k, "fraction": k / len(Cl), "parts holding": sum(parts),
                     "outcome": "reproduced" if all(parts) else ("partially" if any(parts) else "not")})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under both rate-overlap readings, O-OM spatial correlations (median 0.49
against 0.85) and rate overlaps (0.70 and 0.73 against 0.78 and 0.79) are lower than CON2-CON3
values (rank-sum p ≤ 0.00037). Between O and OM 778 and 814 cells (65.4 and 68.5%) change, within
0.697 ± 0.0697.

## figs4-object-traces

> 18.3% of CA1 cells (218 of 1,189) had object-trace fields, firing in NO' where they had fired in
> OM, well above the 1% chance level.

Cells changing between NO and NO' (as above), fields in their NO' map (`field` readings), a
circular ROI per field (claims readings: a, a circle of the field's area at its centroid; b, the
smallest circle at the centroid holding the field), moved to the same place in OM; rate overlap of
the NO' ROI and the OM ROI against 1,000 placements of the ROI at random in OM (wrapped around); a
trace field above the 99th percentile. A cell counts when any field passes.

In [ ]:
SHUF = 1000
NB = oc.nbins(2.5)
TRACES = {}
rows = []
for rm in "b":
    maps = {}
    for si, s in enumerate(CA1):
        if s.trial("NOp") is None:
            continue
        maps[si] = (oc.single_maps(s.trial("NOp"), rm)[0], oc.single_maps(s.trial("OM"), rm)[0])
    for field in "ab":
        for roi in "ab":
            rng = np.random.default_rng(oc.seed_for("figs4-object-traces", rm, field, roi))
            res = {}
            for si, (mN, mM) in maps.items():
                for u in range(CA1[si].n_units):
                    fl = oc.fields(mN[u], 2.5, field)
                    hits = []
                    for idx in fl:
                        iy, ix = oc.roi_offsets(idx, roi)
                        actual = oc.roi_rate_overlap(mN[u], mM[u], iy, ix, NB)
                        sh = rng.integers(0, NB, size=(SHUF, 2))
                        null = oc.roi_rate_overlap_shuffled(mN[u], mM[u], iy, ix, NB, sh)
                        hits.append((idx, iy, ix, actual > np.nanpercentile(null, 99)))
                    res[si, u] = hits
            for r in "ab":
                changed = set(map(tuple, CLASS[rm].loc[CLASS[rm][f"NO-NOp any_{r}"].fillna(False).astype(bool), ["session", "unit"]].values))
                trace = {cu: [h for h in res[cu] if h[3]] for cu in changed if cu in res}
                trace = {cu: h for cu, h in trace.items() if h}
                TRACES[rm, field, roi, r] = trace
                k = len(trace)
                bt = stats.binomtest(k, 1189, 0.01)
                rows.append({"rm": rm, "field": field, "roi": roi, "rate overlap": r, "changed NO-NOp": len(changed),
                             "with NO' field": sum(1 for cu in changed if res.get(cu)), "trace cells": k,
                             "fraction": k / 1189, "binomial p": bt.pvalue,
                             "changed ok": abs(len(changed) - 919) <= 91.9, "fraction ok": abs(k / 1189 - 0.183) <= 0.0183})
figs4 = pd.DataFrame(rows)
figs4["reproduced"] = figs4["changed ok"] & figs4["fraction ok"] & (figs4["binomial p"] < 0.05)
print(figs4.to_string(index=False))
print("\nPer rm and field, any ROI and rate-overlap reading meeting the criterion:")
print(figs4.groupby(["rm", "field"]).reproduced.any().to_string())
print("\nTrace counts against the paper's 18.3% (218 of 1189), two-sided binomial test:")
for _, row in figs4.iterrows():
    bt = stats.binomtest(int(row["trace cells"]), 1189, 0.183)
    print(f"  field {row['field']}, ROI {row['roi']}, rate overlap {row['rate overlap']}: {row['trace cells']} cells, "
          f"{row['trace cells'] / 218:.2f} of the paper's 218, p = {bt.pvalue:.1e}")
print(f"({time.time() - started:.0f} s so far)")

**Result: not reproduced.** 908 and 916 cells change between NO and NO', within 919 ± 91.9.
Under every reading the cells with an object-trace field, 72 to 117 (6.1 to 9.8% of 1,189), are far
above the 1% chance level (binomial p < 1e-32), but fewer than 18.3 ± 1.83%. The trace counts are
0.33 to 0.54 of the paper's 218 and differ from 18.3% by a binomial test under every reading (p at
most 1.4e-16), so the miss is far outside sampling error, unlike fig1c-object-tuned-fraction. No
field, ROI and rate-overlap reading meets the criterion, which has no partial case. Changed after
round 3 from partially reproduced, as the verifier agreed. Only 814 to 886 of the changing cells
have a NO' field, against all 919 in the paper, so the field detection probably finds fewer fields
than the paper's; but among cells with fields the trace fraction is still 10 to 14%, against the
paper's 23.7% (218 of 919). If the question on field detection is answered, this outcome should be
checked again.

## figs2d-traces-no-interference

> Removing object traces from the OM trial did not significantly change the number of object-tuned
> cells (256 of 1,189 after removal, against the expected 19.9%).

The trace ROIs of figs4-object-traces, moved onto the 3 cm template maps of OM (bins whose centre
lies inside the ROI's circle). The claims file's readings: (a) those bins left out of the OM
correlations; (b) the rate there set to zero. Template matching is re-run for the cells with traces;
the others keep their result. A two-sided binomial test of the count against 19.9%.

In [ ]:
X3, Y3 = oc.bin_centres(3.0)
X25, Y25 = oc.bin_centres(2.5)


def roi_mask_3cm(hits, roi):
    m = np.zeros(2500, bool)
    for idx, iy, ix, _ in hits:
        cx, cy = X25[idx].mean(), Y25[idx].mean()
        rad = np.sqrt(len(idx) * 2.5 ** 2 / np.pi) if roi == "a" else np.hypot(X25[idx] - cx, Y25[idx] - cy).max()
        m |= np.hypot(X3 - cx, Y3 - cy) <= rad
    return m.reshape(50, 50)


rows = []
for rm in "b":
    for w in "ab":
        base = oc.object_tuning(CA1, rm, oc.WIDTHS[w], key="CA1")
        base_tuned = {(c["session"], c["unit"]): c["tuned"] for c in base}
        n0 = sum(base_tuned.values())
        for field in "ab":
            for roi in "ab":
                for r in "ab":
                    trace = TRACES[rm, field, roi, r]
                    masks = {cu: roi_mask_3cm(h, roi) for cu, h in trace.items()}
                    sess_ids = sorted({cu[0] for cu in masks})
                    for removal in "ab":
                        sub = [CA1[i] for i in sess_ids]
                        local = {(sess_ids.index(cu[0]), cu[1]): m for cu, m in masks.items()}
                        rerun = oc.object_tuning(sub, rm, oc.WIDTHS[w], key=f"traces-{removal}",
                                                 masks_OM=local if removal == "a" else None,
                                                 zero_OM=local if removal == "b" else None, only=set(local))
                        new = dict(base_tuned)
                        for c in rerun:
                            cu = (sess_ids[c["session"]], c["unit"])
                            if cu in masks:
                                new[cu] = c["tuned"]
                        k = sum(new.values())
                        bt = stats.binomtest(k, 1189, 0.199)
                        rows.append({"rm": rm, "w": w, "field": field, "roi": roi, "rate overlap": r, "removal": removal,
                                     "trace cells": len(masks), "tuned before": n0, "tuned after": k,
                                     "fraction after": k / 1189, "binomial p vs 0.199": bt.pvalue})
figs2d = pd.DataFrame(rows)
print(figs2d.to_string(index=False))
print("\nReadings with p >= 0.05:", int((figs2d["binomial p vs 0.199"] >= 0.05).sum()), "of", len(figs2d))
print(figs2d.groupby(["rm", "w"])["binomial p vs 0.199"].agg(["min", "max"]).to_string())
print(f"\nTotal time {time.time() - started:.0f} s")

**Result: blocked (methods underspecified).** The paper's test gives different outcomes under the
template-width readings. With the Methods' variances (`w` b) the count after removal is 226 to 240
(19.0 to 20.2%), and the binomial test against 19.9% gives p = 0.47 to 1 under every trace and
removal reading, so the claim holds. With the code's SDs (`w` a) it is 340 to 355 (28.6 to 29.9%),
p < 1e-12, a level that reading gives before any removal too (331 tuned). Removing traces raises the
count by 4 to 24 cells. The re-run uses new shuffles for the 72 to 117 trace cells (table above).

## Summary

| Claim | Outcome |
|---|---|
| fig2-fraction-changed-no-o | reproduced (66.4 and 66.7%) |
| fig2d-o-om-change | reproduced |
| figs4-object-traces | not reproduced (trace fields 6.1 to 9.8%, above chance but under half the paper's 18.3%) |
| figs2d-traces-no-interference | blocked, methods underspecified (template width readings disagree) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the random groups of sections are shared by the cells
of a session; the pairing of the signed-rank test is the claims file's (by repetition); the ROI of
a trace field, which the Methods define in a section the paper does not contain, follows the
claims file's two readings.

What was found about the dataset: 99 of 1,189 cells (8.3%) have no No Object' trial, close to the
paper's 8.5%.